# 03. asyncio：从 `async with` 到 `TaskGroup`

这一章是一门完整的小课程，目标是建立两个相互配合的心智模型：

- **`async with` 管资源生命周期**：资源何时打开、出现异常或取消时如何可靠关闭；
- **`TaskGroup` 管任务生命周期**：子任务从哪里产生、何时结束、失败时谁负责取消和回收。

学完后，你应该能解释 `async with` 的协议、实现自己的异步上下文管理器，并用结构化并发组织一组有共同生命周期的任务。

## 课程路线

1. 复习协程、`await` 和并发启动；
2. 从普通 `with` 推导 `async with`；
3. 用类和 `@asynccontextmanager` 管理异步资源；
4. 从手动 Task、`gather()` 的管理成本引出 `TaskGroup`；
5. 深入任务结果、动态添加、取消、`ExceptionGroup` 和嵌套组；
6. 把异步资源与 TaskGroup 组合成完整业务结构。

所有 I/O 都由短暂的 `asyncio.sleep()` 离线模拟。Jupyter 已有事件循环，所以单元格使用顶层 `await`；普通脚本应使用 `asyncio.run(main())`。

## 1. 协程函数、协程对象与 Task

`async def` 定义协程函数。调用它不会立即执行函数体，而是返回协程对象。协程对象需要被 `await`，或者被包装成 Task 交给事件循环调度。

Task 可以理解为“已经安排到事件循环中的协程执行实例”，它保存运行状态、结果、异常和取消状态。

In [18]:
import asyncio
import inspect
from time import perf_counter


async def answer() -> int:
    return 42


coroutine = answer()
print("协程对象：", inspect.iscoroutine(coroutine))
coroutine.close()  # 这里只观察对象，不执行它，主动关闭以避免警告

print("await 的结果：", await answer())

协程对象： True
await 的结果： 42


## 2. `await` 与事件循环

`await` 表示当前协程暂时无法继续。事件循环会保存暂停位置，转去推进其他可运行任务，等等待对象完成后再恢复当前协程。

- `time.sleep(1)`：阻塞事件循环所在的线程，其他协程也无法运行；
- `await asyncio.sleep(1)`：只暂停当前协程，事件循环仍能推进其他任务；
- `await` 是潜在的任务切换点，因此共享状态可能在前后发生变化。

异步不是自动多线程。`asyncio` 默认在一个线程内以协作方式调度多个协程。

In [19]:
async def greet(name: str) -> str:
    await asyncio.sleep(0.05)
    return f"你好，{name}"


print(await greet("asyncio"))

你好，asyncio


## 3. 连续 `await` 仍然串行

如果先等待 A 完成才创建 B，就没有并发。要让等待时间重叠，必须先把多个协程交给事件循环，例如使用 `gather()`、Task 或稍后介绍的 TaskGroup。

In [20]:
async def simulated_request(name: str, delay: float = 0.15) -> str:
    await asyncio.sleep(delay)
    return name


started = perf_counter()
sequential_results = [
    await simulated_request(name) for name in ("A", "B", "C")
]
sequential_elapsed = perf_counter() - started

started = perf_counter()
gather_results = await asyncio.gather(
    simulated_request("A"),
    simulated_request("B"),
    simulated_request("C"),
)
gather_elapsed = perf_counter() - started

assert sequential_results == gather_results
print(f"连续 await：{sequential_elapsed:.2f} 秒")
print(f"gather 并发：{gather_elapsed:.2f} 秒")

连续 await：0.46 秒
gather 并发：0.15 秒


## 4. 从普通 `with` 开始

上下文管理器把“进入—使用—退出”固定成一个作用域。文件、锁和事务都适合这种结构。

```python
with manager as resource:
    use(resource)
```

普通上下文管理器实现：

- `__enter__()`：进入作用域并返回 `as` 后绑定的对象；
- `__exit__(exc_type, exc_value, traceback)`：无论正常结束还是抛出异常都会调用；
- `__exit__()` 返回真值会抑制异常，返回假值或 `None` 会继续传播异常。

Java 对照：接近实现 `AutoCloseable` 后使用 try-with-resources。

In [21]:
class SyncResource:
    def __enter__(self) -> "SyncResource":
        print("1. 同步资源进入")
        return self

    def use(self) -> None:
        print("2. 同步资源使用")

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        print("3. 同步资源退出，异常类型：", exc_type)
        return False


with SyncResource() as sync_resource:
    sync_resource.use()

1. 同步资源进入
2. 同步资源使用
3. 同步资源退出，异常类型： None


`with` 可以近似理解为下面的协议调用：

```python
manager = expression
resource = manager.__enter__()
try:
    # with 代码块
    ...
except BaseException as error:
    suppress = manager.__exit__(type(error), error, error.__traceback__)
    if not suppress:
        raise
else:
    manager.__exit__(None, None, None)
```

关键价值不是少写一次 `close()`，而是把清理动作绑定到语法作用域，所有退出路径都经过统一出口。

## 5. 为什么需要 `async with`

普通 `__enter__()` 和 `__exit__()` 不能使用 `await`。如果打开数据库连接、获取远程租约或关闭网络会话本身也需要等待，就需要异步上下文管理器：

- `async def __aenter__(self)` 返回 awaitable；
- `async def __aexit__(self, exc_type, exc_value, traceback)` 返回 awaitable；
- `async with` 会分别等待这两个方法。

因此，`async with` 不是“让 with 并发执行”，而是允许**进入和退出资源作用域的过程异步等待**。

In [22]:
class CourseConnection:
    def __init__(self, name: str, events: list[str]) -> None:
        self.name = name
        self.events = events
        self.is_open = False

    async def __aenter__(self) -> "CourseConnection":
        self.events.append(f"{self.name}:opening")
        await asyncio.sleep(0.02)
        self.is_open = True
        self.events.append(f"{self.name}:opened")
        return self

    async def query(self, value: str) -> str:
        if not self.is_open:
            raise RuntimeError("连接尚未打开")
        self.events.append(f"query:{value}")
        await asyncio.sleep(0.03)
        return f"result:{value}"

    async def __aexit__(self, exc_type, exc_value, traceback) -> bool:
        exception_name = exc_type.__name__ if exc_type else "None"
        self.events.append(f"{self.name}:closing:{exception_name}")
        await asyncio.sleep(0.02)
        self.is_open = False
        self.events.append(f"{self.name}:closed")
        return False


connection_events: list[str] = []
async with CourseConnection("demo", connection_events) as connection:
    connection_result = await connection.query("Python")

print(connection_result)
print(connection_events)
assert connection_events[-1] == "demo:closed"

result:Python
['demo:opening', 'demo:opened', 'query:Python', 'demo:closing:None', 'demo:closed']


`async with` 的协议可以近似展开为：

```python
manager = expression
resource = await manager.__aenter__()
try:
    # async with 代码块
    ...
except BaseException as error:
    suppress = await manager.__aexit__(type(error), error, error.__traceback__)
    if not suppress:
        raise
else:
    await manager.__aexit__(None, None, None)
```

真实语义由 Python 语言规范定义；这段展开代码用于建立心智模型。注意 `__aexit__()` 能看到离开作用域的异常，并在资源关闭完成后决定是否继续传播。

## 6. 正常退出、异常与抑制

资源管理器常根据 `exc_type` 决定提交还是回滚。通常应返回 `False` 让异常继续传播；只有管理器明确负责处理某类异常时才返回 `True`。

不要为了让日志“看起来没有错误”而随意抑制异常，否则调用者可能误以为操作成功。

In [23]:
class ExpectedBusinessError(Exception):
    pass


class AsyncTransaction:
    async def __aenter__(self) -> "AsyncTransaction":
        print("事务开始")
        await asyncio.sleep(0.01)
        return self

    async def __aexit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            print("事务提交")
        else:
            print(f"事务回滚：{exc_type.__name__}")
        await asyncio.sleep(0.01)
        return exc_type is ExpectedBusinessError


async with AsyncTransaction():
    raise ExpectedBusinessError("该异常由事务管理器处理")
print("ExpectedBusinessError 已被显式抑制")

try:
    async with AsyncTransaction():
        raise RuntimeError("未知错误应继续传播")
except RuntimeError as error:
    print("调用者捕获：", error)

事务开始
事务回滚：ExpectedBusinessError
ExpectedBusinessError 已被显式抑制
事务开始
事务回滚：RuntimeError
调用者捕获： 未知错误应继续传播


## 7. 用 `@asynccontextmanager` 简化实现

只需要管理简单资源时，可以用 `contextlib.asynccontextmanager` 装饰异步生成器：

- `yield` 之前相当于 `__aenter__()`；
- `yield` 的值绑定到 `as` 后的变量；
- `yield` 之后的 `finally` 相当于 `__aexit__()`；
- 每次使用都应调用工厂函数创建新的上下文管理器实例。

In [24]:
from contextlib import asynccontextmanager
from collections.abc import AsyncIterator


@asynccontextmanager
async def managed_session(name: str, events: list[str]) -> AsyncIterator[dict[str, str]]:
    events.append(f"{name}:acquiring")
    await asyncio.sleep(0.02)
    session = {"name": name}
    events.append(f"{name}:acquired")
    try:
        yield session
    finally:
        events.append(f"{name}:releasing")
        await asyncio.sleep(0.02)
        events.append(f"{name}:released")


session_events: list[str] = []
async with managed_session("session-A", session_events) as session:
    print("正在使用：", session)

print(session_events)

正在使用： {'name': 'session-A'}
['session-A:acquiring', 'session-A:acquired', 'session-A:releasing', 'session-A:released']


### 取消时也必须清理

取消会在协程的下一个等待点抛出 `asyncio.CancelledError`。异步上下文管理器仍会退出，因此清理逻辑应放在 `__aexit__()` 或 `finally` 中。

清理完成后应继续传播取消。`TaskGroup` 和 `asyncio.timeout()` 内部都依赖取消机制；吞掉 `CancelledError` 会破坏它们的行为。

In [25]:
cancellation_events: list[str] = []


@asynccontextmanager
async def cancellable_resource() -> AsyncIterator[str]:
    cancellation_events.append("resource:opened")
    try:
        yield "resource"
    finally:
        cancellation_events.append("resource:cleanup-started")
        await asyncio.sleep(0.02)
        cancellation_events.append("resource:closed")


async def long_operation() -> None:
    async with cancellable_resource():
        cancellation_events.append("operation:started")
        await asyncio.sleep(1)


try:
    async with asyncio.timeout(0.05):
        await long_operation()
except TimeoutError:
    cancellation_events.append("timeout:caught")

print(cancellation_events)
assert cancellation_events == [
    "resource:opened",
    "operation:started",
    "resource:cleanup-started",
    "resource:closed",
    "timeout:caught",
]

['resource:opened', 'operation:started', 'resource:cleanup-started', 'resource:closed', 'timeout:caught']


## 8. 从手动 Task 管理到 `gather()`

`asyncio.create_task()` 会立即把协程安排到事件循环。直接创建 Task 很灵活，但父协程必须自行负责：

- 保存引用，避免任务生命周期不清晰；
- 等待结果并读取异常；
- 父任务失败或取消时，取消并回收所有子任务；
- 确保函数返回时没有仍在运行的孤儿任务。

下面的清理模板说明了手动管理为什么容易遗漏细节。

In [28]:
async def manually_managed_tasks() -> list[str]:
    tasks = [
        asyncio.create_task(simulated_request(name, 0.05), name=f"request-{name}")
        for name in ("A", "B", "C")
    ]
    try:
        return [await task for task in tasks]
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)


print(await manually_managed_tasks())

['A', 'B', 'C']


`asyncio.gather()` 能方便地并发等待一组 awaitable，并按输入顺序返回结果。但默认情况下，一个任务抛出异常后，`gather()` 会把第一个异常传播给调用者，其他任务不会因此自动取消。

这不是错误，而是一种明确的失败语义；如果业务要求“一个失败，整组停止”，TaskGroup 更合适。

In [29]:
gather_failure_events: list[str] = []


async def gather_failure() -> None:
    await asyncio.sleep(0.02)
    raise ValueError("gather 中的任务失败")


async def gather_survivor() -> str:
    await asyncio.sleep(0.06)
    gather_failure_events.append("survivor:finished")
    return "survivor result"


survivor_task = asyncio.create_task(gather_survivor())
try:
    await asyncio.gather(gather_failure(), survivor_task)
except ValueError as error:
    gather_failure_events.append(f"caught:{error}")

# gather 已抛出异常，但兄弟任务仍然继续；显式等待以免遗留后台任务。
survivor_result = await survivor_task
print(gather_failure_events, survivor_result)
assert "survivor:finished" in gather_failure_events

['caught:gather 中的任务失败', 'survivor:finished'] survivor result


## 9. `TaskGroup` 的来龙去脉：结构化并发

结构化并发要求任务生命周期与代码块结构一致：父任务创建子任务，离开作用域前必须确认所有子任务已经完成或取消。

`asyncio.TaskGroup` 本身就是一个异步上下文管理器：

- `__aenter__()` 激活任务组；
- `group.create_task()` 创建属于该组的子任务；
- `__aexit__()` 等待所有任务，并统一处理失败与取消；
- 离开 `async with` 后，可以安全读取成功任务的 `result()`。

这解决的不是“如何更快”，而是“谁拥有这些任务、谁负责收尾”。

In [30]:
async def basic_task_group() -> list[str]:
    tasks: list[asyncio.Task[str]] = []
    async with asyncio.TaskGroup() as group:
        for name in ("A", "B", "C"):
            tasks.append(
                group.create_task(
                    simulated_request(name, 0.05),
                    name=f"group-request-{name}",
                )
            )
        print("组内：任务已创建，TaskGroup 尚未退出")

    print("组外：所有任务已经结束")
    return [task.result() for task in tasks]


print(await basic_task_group())

组内：任务已创建，TaskGroup 尚未退出
组外：所有任务已经结束
['A', 'B', 'C']


### TaskGroup 中可以动态添加任务

只要任务组仍处于活动状态，组内任务也可以获得 `group` 引用并创建新的子任务。退出作用域时，TaskGroup 会等待动态加入的任务。

这种能力适合递归发现工作，但仍应设置业务边界，避免任务无限产生。

In [31]:
async def add_discovered_task(
    group: asyncio.TaskGroup,
    tasks: list[asyncio.Task[str]],
) -> None:
    await asyncio.sleep(0.01)
    tasks.append(group.create_task(simulated_request("discovered", 0.04)))


async def dynamic_task_group() -> list[str]:
    result_tasks: list[asyncio.Task[str]] = []
    async with asyncio.TaskGroup() as group:
        result_tasks.append(group.create_task(simulated_request("initial", 0.03)))
        group.create_task(add_discovered_task(group, result_tasks))
    return [task.result() for task in result_tasks]


print(await dynamic_task_group())

['initial', 'discovered']


## 10. 一个子任务失败时发生什么

TaskGroup 中第一个非取消异常会触发以下流程：

1. 不再接受新的任务；
2. 取消尚未完成的兄弟任务；
3. 等待兄弟任务运行各自的 `finally` 清理；
4. 把一个或多个错误组合成 `ExceptionGroup`；
5. 在 `async with` 出口抛出异常组。

`CancelledError` 继承自 `BaseException`。可以捕获它记录日志或清理，但通常必须在清理后 `raise`，让取消继续传播。

In [32]:
task_group_failure_events: list[str] = []


async def group_worker(name: str, delay: float, fail: bool = False) -> None:
    task_group_failure_events.append(f"{name}:started")
    try:
        await asyncio.sleep(delay)
        if fail:
            task_group_failure_events.append(f"{name}:raised")
            raise ValueError(f"{name} 失败")
        task_group_failure_events.append(f"{name}:finished")
    except asyncio.CancelledError:
        task_group_failure_events.append(f"{name}:cancelled")
        raise
    finally:
        task_group_failure_events.append(f"{name}:cleanup")


try:
    async with asyncio.TaskGroup() as group:
        group.create_task(group_worker("failing", 0.02, True))
        group.create_task(group_worker("slow", 0.5))
except* ValueError as errors:
    print("捕获异常：", [str(error) for error in errors.exceptions])

print(task_group_failure_events)
assert "slow:cancelled" in task_group_failure_events
assert "slow:finished" not in task_group_failure_events
assert task_group_failure_events.index("slow:cancelled") < task_group_failure_events.index(
    "slow:cleanup"
)

捕获异常： ['failing 失败']
['failing:started', 'slow:started', 'failing:raised', 'failing:cleanup', 'slow:cancelled', 'slow:cleanup']


### `ExceptionGroup` 与 `except*`

并发任务可能在同一轮中产生多个错误，单个 `except` 无法完整表达。`ExceptionGroup` 保存多个异常，`except*` 会按类型匹配其中的子异常。

不要把 `except*` 理解为“循环执行 except”；它会把匹配的异常子组交给对应分支，未匹配异常仍继续传播。

In [ ]:
release_errors = asyncio.Event()
handled_error_types: list[str] = []


async def coordinated_error(error: Exception) -> None:
    await release_errors.wait()
    raise error


try:
    async with asyncio.TaskGroup() as group:
        group.create_task(coordinated_error(ValueError("值错误")))
        group.create_task(coordinated_error(TypeError("类型错误")))
        await asyncio.sleep(0)  # 让两个任务都开始等待 Event
        release_errors.set()
except* ValueError as errors:
    handled_error_types.extend(type(error).__name__ for error in errors.exceptions)
except* TypeError as errors:
    handled_error_types.extend(type(error).__name__ for error in errors.exceptions)

print("分类处理：", handled_error_types)
assert sorted(handled_error_types) == ["TypeError", "ValueError"]

## 11. 嵌套 TaskGroup：任务形成树

复杂业务可以让每个分支拥有自己的内部 TaskGroup。这样任务形成父子树：外层负责分支，分支负责自己的子任务。

如果内外层同时发生失败，内层组会先整理自己的子任务和异常，然后外层组再处理分支失败。不要依赖异常发生的偶然时间顺序，应按作用域边界设计清理逻辑。

In [ ]:
async def branch(branch_name: str) -> list[str]:
    tasks: list[asyncio.Task[str]] = []
    async with asyncio.TaskGroup() as inner_group:
        for item in ("1", "2"):
            tasks.append(
                inner_group.create_task(
                    simulated_request(f"{branch_name}-{item}", 0.03)
                )
            )
    return [task.result() for task in tasks]


async def nested_groups() -> list[list[str]]:
    branch_tasks: list[asyncio.Task[list[str]]] = []
    async with asyncio.TaskGroup() as outer_group:
        branch_tasks.append(outer_group.create_task(branch("left")))
        branch_tasks.append(outer_group.create_task(branch("right")))
    return [task.result() for task in branch_tasks]


print(await nested_groups())

## 12. 综合案例：外层管资源，内层管任务

最常见的可靠结构是：

```python
async with open_resource() as resource:
    async with asyncio.TaskGroup() as group:
        group.create_task(use(resource))
        group.create_task(use(resource))
```

作用域嵌套顺序非常重要：TaskGroup 先退出并等待所有使用者结束，外层资源随后关闭。若顺序反过来，资源可能在子任务仍使用它时提前关闭。

In [ ]:
async def resource_and_group_demo() -> tuple[list[str], list[str]]:
    events: list[str] = []
    tasks: list[asyncio.Task[str]] = []

    async with CourseConnection("shared", events) as shared_connection:
        async with asyncio.TaskGroup() as group:
            for value in ("A", "B", "C"):
                tasks.append(group.create_task(shared_connection.query(value)))
        results = [task.result() for task in tasks]

    return events, results


combined_events, combined_results = await resource_and_group_demo()
print("结果：", combined_results)
print("生命周期：", combined_events)

closing_index = combined_events.index("shared:closing:None")
assert all(combined_events.index(f"query:{value}") < closing_index for value in ("A", "B", "C"))
assert combined_events[-1] == "shared:closed"

## 13. `gather()` 与 TaskGroup 如何选择

| 维度 | `asyncio.gather()` | `asyncio.TaskGroup` |
|---|---|---|
| 主要用途 | 并发等待一组已知 awaitable | 管理一个作用域内的子任务生命周期 |
| 成功结果 | 直接返回按输入排序的列表 | 保存 Task，退出组后读取 `result()` |
| 默认失败语义 | 传播首个异常，其他任务通常继续 | 首个非取消错误触发兄弟任务取消 |
| 动态添加任务 | 不适合 | 组活动期间可以继续添加 |
| 任务归属 | 调用者自行管理 | 明确属于 TaskGroup 作用域 |
| 推荐场景 | 所有任务应独立完成、需要聚合结果 | 一组任务共同成功或失败、必须统一收尾 |

Java 概念映射：

- `async with` 接近异步版 try-with-resources；
- Task 接近由事件循环调度的 Future，但不是线程；
- `gather()` 可类比聚合多个 `CompletableFuture`；
- TaskGroup 对应结构化并发中的父作用域管理子任务思想。

## 14. 超时也是一种结构化取消

`asyncio.timeout()` 同样是异步上下文管理器。超过期限时，它取消当前任务，在作用域外把取消转换为 `TimeoutError`。

因此超时、TaskGroup 和异步资源清理可以自然嵌套：超时触发取消，TaskGroup 回收子任务，最外层 `async with` 最后关闭资源。

In [ ]:
timeout_events: list[str] = []


async def timeout_worker() -> None:
    try:
        timeout_events.append("worker:started")
        await asyncio.sleep(1)
    finally:
        timeout_events.append("worker:cleanup")


try:
    async with asyncio.timeout(0.05):
        async with asyncio.TaskGroup() as group:
            group.create_task(timeout_worker())
except TimeoutError:
    timeout_events.append("timeout:caught")

print(timeout_events)
assert timeout_events == ["worker:started", "worker:cleanup", "timeout:caught"]

## 15. 用 Semaphore 限制并发量

TaskGroup 负责生命周期，不负责限制同时运行多少个任务。即使创建一万个 Task，也可能只允许三个同时访问数据库。`asyncio.Semaphore` 用于控制进入受限资源的并发数量。

In [ ]:
active_workers = 0
peak_workers = 0
semaphore = asyncio.Semaphore(3)


async def limited_worker(task_id: int) -> None:
    global active_workers, peak_workers
    async with semaphore:
        active_workers += 1
        peak_workers = max(peak_workers, active_workers)
        await asyncio.sleep(0.04)
        active_workers -= 1


async with asyncio.TaskGroup() as group:
    for task_id in range(9):
        group.create_task(limited_worker(task_id))

print("最大同时运行数：", peak_workers)
assert peak_workers == 3

## 16. 接入阻塞函数

如果现有库只有同步阻塞接口，可以用 `asyncio.to_thread()` 把调用放进工作线程，避免阻塞事件循环。它适合 I/O 型阻塞函数，不会让纯 Python CPU 计算自动获得多核并行。

In [ ]:
from time import sleep


def legacy_blocking_api(value: int) -> int:
    sleep(0.05)
    return value * 2


blocking_results = await asyncio.gather(
    *(asyncio.to_thread(legacy_blocking_api, value) for value in range(3))
)
print(blocking_results)

## 17. 常见误区

- `async with` 不会自动让代码并发；它只是异步等待进入和退出协议。
- `__aexit__()` 返回 `True` 会抑制异常，不应随意使用。
- 连续多个 `await` 仍然串行；并发需要创建多个任务。
- 裸 `create_task()` 不是错误，但调用者必须保存引用、等待异常并负责取消收尾。
- `gather()` 默认不会因一个任务失败而取消其他任务；TaskGroup 会。
- 不要在清理后吞掉 `CancelledError`，TaskGroup 和 timeout 都依赖取消传播。
- TaskGroup 退出前不能安全假设所有结果已经就绪；退出后再调用 Task 的 `result()`。
- 资源作用域应包在 TaskGroup 外层，确保任务结束后才关闭共享资源。
- Jupyter 中已有事件循环，应使用顶层 `await`；普通脚本使用 `asyncio.run()`。

## 18. 分级练习

### 练习 1：实现异步计时器

分别用类和 `@asynccontextmanager` 实现一个异步计时器，确保代码块抛出异常时仍打印耗时。

### 练习 2：观察失败语义

让三个任务中的第二个任务失败，分别使用 `gather()` 和 TaskGroup，记录其他任务是完成还是取消。

### 练习 3：资源池批处理

外层创建模拟连接，内层 TaskGroup 处理 12 个任务，再用 Semaphore 把并发量限制为 3。断言连接关闭事件发生在最后一个任务清理之后。

### 练习 4：异常分类

让一组任务分别抛出 `ValueError` 与 `TypeError`，使用两个 `except*` 分支分类记录，确认没有未处理异常。

完整命令行案例见 `examples/07_async_with_taskgroup.py`。

## 本章小结

- `await` 管一个等待点；
- `async with` 管一个异步资源作用域；
- TaskGroup 管一组子任务作用域；
- Semaphore 管同时访问资源的数量；
- timeout 给整个作用域设置时间边界。

可靠的异步程序并不只是“同时启动很多协程”，而是让资源、任务、错误、取消和超时都拥有清晰的父级和结束位置。下一章将比较 asyncio、AnyIO 与 Trio 对这些概念的不同表达方式。